In [ ]:
# Load FD001, select non-constant sensors, train a small DKL, and run online updater simulation
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

import sys
import warnings
from pathlib import Path
import importlib

# Resolve project root dynamically
import json

PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

warnings.filterwarnings('ignore')

import src.data.data_loader as data_loader_module
import src.models.dkl_autoencoder_svgp as dkl_model_module
import src.models.train_dkl_svgp as train_dkl_module
import src.models.online_updater as online_updater_module

importlib.reload(data_loader_module)
importlib.reload(dkl_model_module)
importlib.reload(train_dkl_module)
importlib.reload(online_updater_module)

load_cmapss_data = data_loader_module.load_cmapss_data
StreamingCMAPSSDataset = data_loader_module.StreamingCMAPSSDataset
get_sensor_feature_columns = data_loader_module.get_sensor_feature_columns
train_dkl_autoencoder = train_dkl_module.train_dkl_autoencoder
simulate_online_stream_and_update = online_updater_module.simulate_online_stream_and_update


def print_prediction_direction_counts(model_name, y_true, y_pred):
    y_true = [int(round(float(value))) for value in y_true]
    y_pred = [int(round(float(value))) for value in y_pred]

    greater_count = sum(predicted > actual for predicted, actual in zip(y_pred, y_true))
    less_count = sum(predicted < actual for predicted, actual in zip(y_pred, y_true))
    equal_count = sum(predicted == actual for predicted, actual in zip(y_pred, y_true))
    total_count = len(y_true)

    print(f"\n{model_name} rounded RUL direction counts:")
    print(f" - Predicted RUL > actual RUL: {greater_count}")
    print(f" - Predicted RUL < actual RUL: {less_count}")
    print(f" - Predicted RUL = actual RUL: {equal_count}")
    print(f" - Total predictions: {total_count}")

# Paths
PROJECT_ROOT = Path('..')
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
TRAIN_FILE = str(DATA_DIR / 'train_FD001.txt')
TEST_FILE = str(DATA_DIR / 'test_FD001.txt')
RUL_FILE = str(DATA_DIR / 'RUL_FD001.txt')

# Load dataframes
print('Loading FD001 data...')
df_train = load_cmapss_data(TRAIN_FILE)
df_test = load_cmapss_data(TEST_FILE, RUL_FILE)

# Automatic sensor selection: remove near-constant sensors (zero variance)
sensor_candidates = get_sensor_feature_columns(df_train)
vars = df_train[sensor_candidates].var()
VAR_THRESHOLD = 1e-8
active_features = vars[vars > VAR_THRESHOLD].index.tolist()
print(f"Selected {len(active_features)} active sensors: {active_features}")

# Prepare scaler and datasets (preserve RUL clipping consistent with pipeline)
MAX_RUL_SCALE = 125.0
scaler = StandardScaler()

train_dataset = StreamingCMAPSSDataset(
    df_train,
    features=active_features,
    scaler=scaler,
    fit_scaler=True,
    fit_target_transform=True,
    max_rul=MAX_RUL_SCALE,
)
# For test dataset, reuse the fitted target transform from train
test_dataset = StreamingCMAPSSDataset(
    df_test,
    features=active_features,
    scaler=scaler,
    fit_scaler=False,
    target_transform=train_dataset.target_transform,
    fit_target_transform=False,
    max_rul=MAX_RUL_SCALE,
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
# Streaming test loader (batch_size=1 to simulate online arrival)
test_stream_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

input_dim = len(active_features)
latent_dim = 8

# Train a compact DKL model (reduce epochs for quick demo)
print('Training DKL Autoencoder SVGP (short run)...')
model, likelihood = train_dkl_autoencoder(
    train_loader=train_loader,
    input_dim=input_dim,
    latent_dim=latent_dim,
    num_inducing=500,
    epochs=8,
    encoder_lr=1e-4,
    decoder_lr=1e-4,
    gp_lr=1e-2,
    lambda_recon=0.1,
    seed=42,
)

# Run the online updater simulation on the FD001 test stream
print('\nRunning online updater simulation on FD001 test stream...')
results = simulate_online_stream_and_update(
    model=model,
    likelihood=likelihood,
    stream_loader=test_stream_loader,
    update_every_x_cycles=10,
    fine_tune_epochs=1,
    lr=0.002,
    lambda_recon=0.1,
    weight_decay=1e-4,
    tune_feature_extractor=False,
    grad_clip_norm=1.0,
    device='cpu',
    collect_metrics=True,
)

print('\nOnline updater completed. Summary metrics:')
if 'metrics' in results:
    for k,v in results['metrics'].items():
        print(f" - {k}: {v}")
else:
    print('No metrics were collected.')

if 'predictions' in results:
    print_prediction_direction_counts(
        'DKL',
        results['predictions']['y_true'],
        results['predictions']['y_pred'],
    )

# add the right prediction cound > d, < d, = d, and total counts for each category


Loading FD001 data...
Selected 15 active sensors: ['s_2', 's_3', 's_4', 's_6', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']
Training DKL Autoencoder SVGP (short run)...
--- Starting DKL Autoencoder SVGP Training (8 Epochs) ---
Epoch 01 | ELBO Loss: 19.9981 | Recon Loss: 0.8667
Epoch 02 | ELBO Loss: 9.4127 | Recon Loss: 0.5416
Epoch 03 | ELBO Loss: 8.3569 | Recon Loss: 0.4382
Epoch 04 | ELBO Loss: 7.6108 | Recon Loss: 0.3923
Epoch 05 | ELBO Loss: 7.0622 | Recon Loss: 0.3557
Epoch 06 | ELBO Loss: 6.5916 | Recon Loss: 0.3219
Epoch 07 | ELBO Loss: 6.2149 | Recon Loss: 0.2917
Epoch 08 | ELBO Loss: 5.8129 | Recon Loss: 0.2661
DKL Autoencoder SVGP Training Evaluation Summary:
  PHM08 Asymmetric Score (Lower is better): 631.6476
  Fleet Correlation Metric (Higher is better): 0.9009
  R2 (Higher is better): 0.8101
  MAE (Mean Absolute Error): 0.3343
  RMSE (Root Mean Squared Error): 0.4358
  Perfect Predictions (d = 0): 0.0000
  Early Predictions (d < 0, 

# RANDOM FOREST BASELINE - NOT ONLINE

In [ ]:
# Compare DKL online-updated model with the Random Forest baseline on FD001
from src.models.baseline import evaluate_baseline_for_subset

print('\nEvaluating baseline for FD001 (Random Forest)...')
baseline = evaluate_baseline_for_subset('FD001')

print('\nComparison (FD001):')
if 'metrics' in results:
    dkl_rmse = results['metrics'].get('rmse')
    dkl_mae = results['metrics'].get('mae')
    baseline_rmse = baseline.get('rmse')
    baseline_mae = baseline.get('mae')

    print(f" - Baseline RMSE: {baseline_rmse:.3f} | DKL (online) RMSE: {dkl_rmse:.3f}")
    print(f" - Baseline MAE:  {baseline_mae:.3f} | DKL (online) MAE:  {dkl_mae:.3f}")

    try:
        rmse_rel = (baseline_rmse - dkl_rmse) / baseline_rmse * 100.0
        mae_rel = (baseline_mae - dkl_mae) / baseline_mae * 100.0
        print(f"\nRelative improvement vs baseline: RMSE {rmse_rel:+.1f}%, MAE {mae_rel:+.1f}%")
    except Exception:
        pass
else:
    print('No DKL metrics available to compare.')



Evaluating baseline for FD001 (Random Forest)...

--- BASELINE BENCHMARK RESULTS (FD001 TEST SET) ---
Sensors used: 21
RMSE: 17.12 cycles
MAE:  12.42 cycles


Comparison (FD001):
 - Baseline RMSE: 17.116 | DKL (online) RMSE: 23.612
 - Baseline MAE:  12.416 | DKL (online) MAE:  19.377

Relative improvement vs baseline: RMSE -38.0%, MAE -56.1%


# ONLINE RANDOM FOREST BASELINE

In [ ]:
# Online RandomForest baseline (periodic retrain to simulate online learning)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import time

print('\nStarting online RandomForest simulation (periodic retrain)...')

# Initialize online-RF (we'll retrain periodically on accumulated data to mimic online behavior)
rf_online = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

# Prepare initial seen dataset from training data
X_train_scaled = scaler.transform(df_train[active_features].values)
y_train_raw = df_train['RUL'].values
X_seen = X_train_scaled.copy()
y_seen = y_train_raw.copy()

rf_online.fit(X_seen, y_seen)

update_every = 30  # match DKL online update frequency
new_count = 0
preds = []
trues = []
latencies = []

for batch_idx, (X_batch, y_batch, unit_nrs) in enumerate(test_stream_loader):
    # X_batch already scaled by StreamingCMAPSSDataset; y_batch is standardized smooth RUL
    X_np = X_batch.numpy()
    y_true_raw = test_dataset.target_transform.inverse_transform(y_batch.numpy().flatten())

    t0 = time.time()
    y_pred = rf_online.predict(X_np)
    t1 = time.time()

    preds.extend(y_pred.tolist())
    trues.extend(np.asarray(y_true_raw).tolist())
    latencies.append(t1 - t0)

    # Accumulate for periodic retrain
    X_seen = np.vstack([X_seen, X_np])
    y_seen = np.concatenate([y_seen, np.asarray(y_true_raw)])
    new_count += len(y_true_raw)

    if new_count >= update_every:
        print(f"Retraining RF after receiving {new_count} new samples (at batch {batch_idx})...")
        rf_online.fit(X_seen, y_seen)
        new_count = 0

# Final metrics
rmse_rf = float(np.sqrt(mean_squared_error(trues, preds)))
mae_rf = float(mean_absolute_error(trues, preds))
print('\nOnline RandomForest simulation completed:')
print(f' - RMSE: {rmse_rf:.3f} | MAE: {mae_rf:.3f} | Avg predict latency: {np.mean(latencies):.4f}s')

print_prediction_direction_counts('Random Forest', trues, preds)

# Compare to DKL results if available
if 'metrics' in results:
    dkl_rmse = results['metrics'].get('rmse')
    dkl_mae = results['metrics'].get('mae')
    print('\nComparison (FD001):')
    print(f' - RF (online) RMSE: {rmse_rf:.3f} | DKL (online) RMSE: {dkl_rmse:.3f}')
    print(f' - RF (online) MAE:  {mae_rf:.3f} | DKL (online) MAE:  {dkl_mae:.3f}')
else:
    print('No DKL metrics found for comparison.')



Starting online RandomForest simulation (periodic retrain)...
Retraining RF after receiving 10 new samples (at batch 9)...
Retraining RF after receiving 10 new samples (at batch 19)...
Retraining RF after receiving 10 new samples (at batch 29)...
Retraining RF after receiving 10 new samples (at batch 39)...
Retraining RF after receiving 10 new samples (at batch 49)...
Retraining RF after receiving 10 new samples (at batch 59)...
Retraining RF after receiving 10 new samples (at batch 69)...
Retraining RF after receiving 10 new samples (at batch 79)...
Retraining RF after receiving 10 new samples (at batch 89)...
Retraining RF after receiving 10 new samples (at batch 99)...
Retraining RF after receiving 10 new samples (at batch 109)...
Retraining RF after receiving 10 new samples (at batch 119)...
Retraining RF after receiving 10 new samples (at batch 129)...
Retraining RF after receiving 10 new samples (at batch 139)...
Retraining RF after receiving 10 new samples (at batch 149)...
Ret

In [1]:
# Load FD001, select non-constant sensors, train a small DKL, and run online updater simulation
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

import sys
import warnings
from pathlib import Path
import importlib

# Resolve project root dynamically
import json

PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

warnings.filterwarnings('ignore')

import src.data.data_loader as data_loader_module
import src.models.dkl_autoencoder_svgp as dkl_model_module
import src.models.train_dkl_svgp as train_dkl_module
import src.models.online_updater as online_updater_module

importlib.reload(data_loader_module)
importlib.reload(dkl_model_module)
importlib.reload(train_dkl_module)
importlib.reload(online_updater_module)

load_cmapss_data = data_loader_module.load_cmapss_data
StreamingCMAPSSDataset = data_loader_module.StreamingCMAPSSDataset
get_sensor_feature_columns = data_loader_module.get_sensor_feature_columns
train_dkl_autoencoder = train_dkl_module.train_dkl_autoencoder
simulate_online_stream_and_update = online_updater_module.simulate_online_stream_and_update


def print_prediction_direction_counts(model_name, y_true, y_pred):
    y_true = [int(round(float(value))) for value in y_true]
    y_pred = [int(round(float(value))) for value in y_pred]

    greater_count = sum(predicted > actual for predicted, actual in zip(y_pred, y_true))
    less_count = sum(predicted < actual for predicted, actual in zip(y_pred, y_true))
    equal_count = sum(predicted == actual for predicted, actual in zip(y_pred, y_true))
    total_count = len(y_true)

    print(f"\n{model_name} rounded RUL direction counts:")
    print(f" - Predicted RUL > actual RUL: {greater_count}")
    print(f" - Predicted RUL < actual RUL: {less_count}")
    print(f" - Predicted RUL = actual RUL: {equal_count}")
    print(f" - Total predictions: {total_count}")

# Paths
PROJECT_ROOT = Path('..')
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
TRAIN_FILE = str(DATA_DIR / 'train_FD001.txt')
TEST_FILE = str(DATA_DIR / 'test_FD001.txt')
RUL_FILE = str(DATA_DIR / 'RUL_FD001.txt')

# Load dataframes
print('Loading FD001 data...')
df_train = load_cmapss_data(TRAIN_FILE)
df_test = load_cmapss_data(TEST_FILE, RUL_FILE)

# Automatic sensor selection: remove near-constant sensors (zero variance)
sensor_candidates = get_sensor_feature_columns(df_train)
vars = df_train[sensor_candidates].var()
VAR_THRESHOLD = 1e-8
active_features = vars[vars > VAR_THRESHOLD].index.tolist()
print(f"Selected {len(active_features)} active sensors: {active_features}")

# Prepare scaler and datasets (preserve RUL clipping consistent with pipeline)
MAX_RUL_SCALE = 125.0
scaler = StandardScaler()

train_dataset = StreamingCMAPSSDataset(
    df_train,
    features=active_features,
    scaler=scaler,
    fit_scaler=True,
    fit_target_transform=True,
    max_rul=MAX_RUL_SCALE,
)
# For test dataset, reuse the fitted target transform from train
test_dataset = StreamingCMAPSSDataset(
    df_test,
    features=active_features,
    scaler=scaler,
    fit_scaler=False,
    target_transform=train_dataset.target_transform,
    fit_target_transform=False,
    max_rul=MAX_RUL_SCALE,
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
# Streaming test loader (batch_size=1 to simulate online arrival)
test_stream_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

input_dim = len(active_features)
latent_dim = 8

# Train a compact DKL model (reduce epochs for quick demo)
print('Training DKL Autoencoder SVGP (short run)...')
model, likelihood = train_dkl_autoencoder(
    train_loader=train_loader,
    input_dim=input_dim,
    latent_dim=latent_dim,
    num_inducing=500,
    epochs=8,
    encoder_lr=1e-4,
    decoder_lr=1e-4,
    gp_lr=1e-2,
    lambda_recon=0.1,
    beta_kl=0.1,
    seed=42,
)

# Run the online updater simulation on the FD001 test stream
print('\nRunning online updater simulation on FD001 test stream...')
results = simulate_online_stream_and_update(
    model=model,
    likelihood=likelihood,
    stream_loader=test_stream_loader,
    update_every_x_cycles=10,
    fine_tune_epochs=1,
    lr=0.002,
    lambda_recon=0.1,
    weight_decay=1e-4,
    tune_feature_extractor=False,
    grad_clip_norm=1.0,
    device='cpu',
    collect_metrics=True,
)

print('\nOnline updater completed. Summary metrics:')
if 'metrics' in results:
    for k,v in results['metrics'].items():
        print(f" - {k}: {v}")
else:
    print('No metrics were collected.')

if 'predictions' in results:
    print_prediction_direction_counts(
        'DKL',
        results['predictions']['y_true'],
        results['predictions']['y_pred'],
    )

# add the right prediction cound > d, < d, = d, and total counts for each category


Loading FD001 data...
Selected 15 active sensors: ['s_2', 's_3', 's_4', 's_6', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']
Training DKL Autoencoder SVGP (short run)...
--- Encoder Warm-up (10 epochs, reconstruction only) ---
  Warm-up epoch 01 | Recon Loss: 0.6760
  Warm-up epoch 02 | Recon Loss: 0.2945
  Warm-up epoch 03 | Recon Loss: 0.1993
  Warm-up epoch 04 | Recon Loss: 0.1613
  Warm-up epoch 05 | Recon Loss: 0.1319
  Warm-up epoch 06 | Recon Loss: 0.1127
  Warm-up epoch 07 | Recon Loss: 0.1049
  Warm-up epoch 08 | Recon Loss: 0.1006
  Warm-up epoch 09 | Recon Loss: 0.0955
  Warm-up epoch 10 | Recon Loss: 0.0929


UnboundLocalError: cannot access local variable 'x' where it is not associated with a value